# AquaHealth AI — Phase 12: CV experiment runner on Colab GPU

One experiment = one model × one fold × one data arm, trained by `scripts/run_cv_experiment.py` on CUDA. This notebook is the only place experiments run; the local machine is for code and manifests.

Rules baked into the code: `final_test.csv` is read for ids only; validation is always `fold_XX_validation.csv`; WITH-GAN data comes from `data/gan/fold_XX/`; a COMPLETED experiment is never retrained; `--require-cuda` aborts without a GPU.

## 1. Runtime → GPU. Clone the repository, install the pinned dependencies

In [ ]:
!nvidia-smi
import os, pathlib, subprocess
REPO_URL = 'https://github.com/kolursamith/aquahealth.git'   # adjust if your remote differs
BRANCH = 'develop'                                            # the single development branch
REPO_DIR = pathlib.Path('/content/aquahealth')
if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log -1 --oneline
!pip install -q -r requirements/experiments.txt   # torch/torchvision/opencv/matplotlib pins; nothing else


## 2. CUDA, GPU smoke test and environment pre-flight (abort here if there is no GPU)

`scripts/gpu_smoke.py` moves a tensor to the GPU and runs one forward/backward/optimizer step of a tiny network — not training. `scripts/colab_preflight.py --env-only` reports Python, torch/torchvision/numpy/pillow/opencv/matplotlib (the project's actual dependencies) and the repository digests. Exit code 2 = no CUDA.

In [ ]:
!python scripts/gpu_smoke.py --require-cuda
!python scripts/colab_preflight.py --env-only --require-cuda


## 3. Dataset root: `AQUAHEALTH_COLAB_DATASET_ROOT` (no Google Drive assumption)

The clean manifest in git (`data/audit/clean_manifest.csv`, 5,942 included images) is the dataset's source of truth; images are never in git and the local Mac path is never used here. Set the environment variable to **one** of:

* the **clean bundle** built locally with `python scripts/build_colab_bundle.py --out <dir> --tar` (≈5.5 GB: exactly the included images at their manifest paths `data/raw/<key>/…`, plus `bundle_manifest.csv` and `bundle.sha256` pinning it to the manifest/split/fold digests) — upload the tar to Drive, a bucket or the VM and extract it; or
* the **delivered `Dataset/` drop** (the five deliveries as delivered) — attached through `scripts/link_raw_datasets.py`.

The cell fails with a clear message if the variable is unset or the folder is missing. Mounting Drive is only one way to bring the files in; uncomment it if that is where you put them.

In [ ]:
import os, pathlib
# --- choose ONE way to bring the data in -------------------------------------------------
# (a) Google Drive (optional): from google.colab import drive; drive.mount('/content/drive')
#     then e.g. !tar -xf /content/drive/MyDrive/AquaHealth/aquahealth_bundle.tar -C /content/aquahealth_data
# (b) any other copy: gsutil/curl/scp into /content/aquahealth_data
# ------------------------------------------------------------------------------------------
os.environ.setdefault('AQUAHEALTH_COLAB_DATASET_ROOT', '/content/aquahealth_data/aquahealth_bundle')
root = pathlib.Path(os.environ['AQUAHEALTH_COLAB_DATASET_ROOT'])
assert root.is_dir(), (
    f"AQUAHEALTH_COLAB_DATASET_ROOT={root} does not exist on this runtime. "
    "Extract the bundle (or place the delivered Dataset/ folder) there, or set the variable to where it is."
)
!python scripts/colab_dataset.py info
!python scripts/colab_dataset.py attach --force
!ls -la data/raw


## 4. Data integrity inside Colab — never silently regenerate

`verify` proves that what is attached is the verified dataset: manifest/split/fold digests, bundle pins (SHA-256 of the clean manifest, the development and frozen-test split files and folds.csv), every included image present, count = 5,942, canonical labels, group ids, fold ids, and image SHA-256 against the manifest (`--hash all`; use `--hash sample` for a quick seeded 200-file check). A non-zero exit means STOP — do not rebuild manifests on Colab.

In [ ]:
!python scripts/colab_dataset.py verify --hash all


## 5. Persistent results and GAN outputs (optional)

If Drive is mounted, `results/v2` and `data/gan` can be symlinked to it so checkpoints, histories and synthetic images survive a disconnect. Skip if you persist them another way.

In [ ]:
PERSIST_ON_DRIVE = False
if PERSIST_ON_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = pathlib.Path('/content/drive/MyDrive/AquaHealth')
    for local, remote in (('results/v2', DRIVE / 'results_v2'), ('data/gan', DRIVE / 'gan')):
        remote.mkdir(parents=True, exist_ok=True)
        local = pathlib.Path(local)
        if local.exists() and not local.is_symlink():
            !rsync -a --ignore-existing "{local}/" "{remote}/"
            !rm -rf "{local}"
        if not local.is_symlink():
            local.symlink_to(remote)
        print(local, '->', local.resolve())


## 6. Pre-flight for the requested experiment (integrity, dataset, manifests, CUDA)

In [ ]:
MODEL = 'cnn_vit_lstm'        # efficientnet_b0 | cnn_vit_lstm | yolo_efficientnet | cnn_bilstm | resnet_attention | yolo_transformer
FOLD = 1                     # 1..10
DATA_ARM = 'without_gan'     # without_gan | with_gan
!python scripts/colab_preflight.py --fold {FOLD} --data-arm {DATA_ARM} --require-cuda

## 7. (WITH-GAN only) generate the fold's synthetic training images first

Trains the fold-specific cDCGAN on `fold_XX_train.csv` only and writes `data/gan/fold_XX/`. Skip for WITHOUT-GAN.

In [ ]:
RUN_GAN = False   # set True for DATA_ARM == 'with_gan' when data/gan/fold_XX does not exist yet
if RUN_GAN:
    !python scripts/run_gan_fold.py --fold {FOLD} --epochs 30 --device cuda
    !python scripts/colab_preflight.py --fold {FOLD} --data-arm with_gan --require-cuda

## 8. INFRASTRUCTURE SMOKE TEST — NOT A PERFORMANCE RESULT

cnn_vit_lstm · fold 01 · WITHOUT-GAN · 64 training / 32 validation images · 2 epochs (1 head + 1 full) on CUDA. Exercises loading, CLAHE, model, forward/backward, optimiser, validation, metrics, checkpoints, resume and result writing. Written under `results/v2/smoke/` (never mixed with the real matrix). Do not report its metrics as accuracy.

In [ ]:
SMOKE_ID = 'smoke_cnn_vit_lstm_fold01_without_gan'
!python scripts/run_cv_experiment.py --model cnn_vit_lstm --fold 1 --data-arm without_gan \
    --config configs/cv_v2/smoke.json --require-cuda --smoke \
    --max-train-samples 64 --max-validation-samples 32 \
    --out-root results/v2/smoke --experiment-id {SMOKE_ID}
!ls -la results/v2/smoke/{SMOKE_ID} && cat results/v2/smoke/{SMOKE_ID}/run_summary.json

## 9. Launch ONE real experiment (Phase 13 — do not run in Phase 12)

Set RUN_EXPERIMENT = True only when the Phase-13 execution plan says so. `--resume` continues an interrupted run; a COMPLETED experiment is refused.

In [ ]:
RUN_EXPERIMENT = False
RESUME = False
if RUN_EXPERIMENT:
    resume_flag = '--resume' if RESUME else ''
    !python scripts/run_cv_experiment.py --model {MODEL} --fold {FOLD} --data-arm {DATA_ARM} \
        --config configs/cv_v2/default.json --require-cuda {resume_flag} \
        2>&1 | tee -a "results/v2/experiments/{MODEL}_fold{FOLD:02d}_{DATA_ARM}.console.log"
    !python scripts/build_experiment_matrix.py --refresh
    !grep -c COMPLETED results/v2/experiment_matrix.csv || true

## 10. After a disconnect

Re-run sections 1–5 (dataset root, verify, optional Drive links), then section 9 with `RESUME = True`. `latest.pt` holds model/optimizer/scheduler/scaler/RNG state and the history; the configuration hash must match.